This example considers a toy problem generated by ChatGPT 5.1 to come up with a problem that may cause scaling issues with parameter size. The model considered is a one state, two input (one fixed, other doe design variable), six parameter system. This model solves for the scaled version:



UNSCALED VERSION
$\dot{x}(t) = -a_1 x(t) + a_2 e^{k_1 u_1(t)} + a_3 e^{k_2 u_2} - a_4 e^{-x(t)}$

PARAMETERS:
1. $a_1 = 1 \in [0.1,2]$
2. $a_2 = 2 \in [0.1,5]$
3. $a_3 = 1.5 \in [0.1,5]$
4. $a_4 = 0.8 \in [0.1,5]$
5. $k_1 = 0.5 \in [-2,2]$
6. $k_2 = 0.3 \in [-2,2]$

STATE AND CONTROL INPUT:
1. $x \in [0,5]$
2. $u_1 \in [0,1]$

FIXED CONTROL INPUT:

1. $u_2 = 1 \in [0,2]$
   
SCALED VERSION

PARAMETERS:
1. $a_1 \sim [0.1,2]$; $\hat{a}_1 = (a_1 - 0.1)/1.9 \sim [0,1]$
2. $a_2 = 2 \in [0.1,5]$; $\hat{a}_2 = (a_2 -0.1)/4.9 \sim [0,1]$
3. $a_3 = 1.5 \in [0.1,5]$; $\hat{a}_3 = (a_3 -0.1)/4.9 \sim [0,1]$
4. $a_4 = 0.8 \in [0.1,5]$; $\hat{a}_4 = (a_4 -0.1)/4.9 \sim [0,1]$
5. $k_1 = 0.5 \in [-2,2]$; $\hat{k}_1 = (k_1 +2)/4 \sim [0,1]$
6. $k_2 = 0.3 \in [-2,2]$; $\hat{k}_2 = (k_2 +2)/4 \sim [0,1]$

STATE AND CONTROL INPUT:
1. $x \sim [0,5]$; $\hat{x} = \frac{x}{5}\sim[0,1]$
2. $u_1 \in [0,1]$

FIXED CONTROL INPUT:

1. $u_2 = 1 \in [0,2]$; $\hat{u}_2 = u_2/2 \sim [0,1]$; $\hat{u}_2 = 0.5$


$\dot{\hat{x}}(t) = -(1.9\hat{a}_1+0.1) \hat{x}(t) + \frac{(4.9\hat{a}_2+0.1)}{5} e^{(4\hat{k}_1-2) u_1(t)} + \frac{(4.9\hat{a}_3+0.1)}{5} e^{(4\hat{k}_2-2) 2\hat{u}_2} - \frac{(4.9\hat{a}_4+0.1)}{5} e^{-5\hat{x}(t)}$



In [1]:

from pyomo.dae import ContinuousSet, DerivativeVar, Simulator
import numpy as np
from pathlib import Path
from pyomo.contrib.parmest.experiment import Experiment



# ========================
class SixParameterExperiment(Experiment):
    def __init__(self, data, nfe, ncp):
        self.data = data
        self.nfe = nfe
        self.ncp = ncp
        self.model = None


    def get_labeled_model(self):
        if self.model is None:
            self.create_model()
            self.finalize_model()
            self.label_experiment()
        return self.model

   
    def create_model(self):
     

        m = self.model = pyo.ConcreteModel()

        # Model parameters
        m.w = pyo.Param(mutable=False, initialize=1.0)

       
        m.t = ContinuousSet(bounds=[0, 1])

        # State and input
        m.x = pyo.Var(m.t, within=pyo.Reals, bounds = [0,1])
        m.u1 = pyo.Var(m.t, within=pyo.Reals, bounds = [0,1])
        m.u2 = pyo.Param(mutable=False, initialize=0.5)

        # Unknown parameter
        m.a1 = pyo.Var(within=pyo.Reals, bounds = [0,1])
        m.a2 = pyo.Var(within=pyo.Reals, bounds = [0,1])
        m.a3 = pyo.Var(within=pyo.Reals, bounds = [0,1])
        m.a4 = pyo.Var(within=pyo.Reals, bounds = [0,1])
        m.k1 = pyo.Var(within=pyo.Reals, bounds = [0,1])
        m.k2 = pyo.Var(within=pyo.Reals, bounds = [0,1])

        # Differential variables wrt x
        m.dxdt = DerivativeVar(m.x, wrt=m.t)
       

    

        # State odes
        @m.Constraint(m.t)
        def x_ode(m, t):
            return m.dxdt[t] == -((1.9*m.a1 + 0.1 )/5) * m.x[t] + ( (4.9*m.a2 + 0.1)/5 )* pyo.exp((4*m.k1-2)*m.u1[t])
            + ( (4.9*m.a3 + 0.1)/5 )*pyo.exp(4*(2*m.k2-1)*m.u2) - ( (4.9*m.a4 + 0.1)/5 )**pyo.exp(-5*m.x[t])
            


    def finalize_model(self):
      
        m = self.model

      
        control_points = self.data["control_points"]

      
        m.x[0].value = self.data["x0"] # Initial state

        # Update model time `t` with time range and control time points
        m.t.update(self.data["t_range"])
        m.t.update(control_points)

        # Fix the unknown parameter values
        m.a1.fix(self.data["a1"])
        m.a2.fix(self.data["a2"])
        m.a3.fix(self.data["a3"])
        m.a4.fix(self.data["a4"])
        m.k1.fix(self.data["k1"])
        m.k2.fix(self.data["k2"])
        # m.u2.fix()
        

        m.t_control = control_points

        # Discretizing the model
        discr = pyo.TransformationFactory("dae.collocation")
        discr.apply_to(m, nfe=self.nfe, ncp=self.ncp, wrt=m.t)

        # Initializing control input in the model
        cv = None
        for t in m.t:
            if t in control_points:
                cv = control_points[t]
                m.u1[t].fix()
            m.u1[t].setlb(self.data["u_bounds"][0])
            m.u1[t].setub(self.data["u_bounds"][1])
            m.u1[t] = cv

        # Make a constraint that holds control input constant between control time points
        @m.Constraint(m.t - control_points)
        def u_control(m, t):
            """
            Piecewise constant control input between control points
            """
            neighbour_t = max(tc for tc in control_points if tc < t)
            return m.u1[t] == m.u1[neighbour_t]


        

        #########################
        # End model finalization

    def label_experiment(self):
        """
        Example for annotating (labeling) the model with a
        full experiment.
        """
        m = self.model

        # Set measurement labels
        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        # Add CA to experiment outputs
        m.experiment_outputs.update((m.x[t], None) for t in m.t_control)


        # Adding error for measurement values (assuming no covariance and constant error for all measurements)
        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        meas_error = 1e-2  # Error in state measurement

        m.measurement_error.update((m.x[t], meas_error) for t in m.t_control)
   

        # Identify design variables (experiment inputs) for the model
        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
    
        # Add experimental input label for control input
        m.experiment_inputs.update((m.u1[t], None) for t in m.t_control)

        # Add unknown parameter labels
        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        # Add labels to all unknown parameters with nominal value as the value
        m.unknown_parameters.update((k, pyo.value(k)) for k in [m.a1, m.a2, m.a3, m.a4, m.k1, m.k2])

        #########################
        # End model labeling

In [2]:
from pyomo.common.dependencies import numpy as np

from pyomo.contrib.doe import DesignOfExperiments

import pyomo.environ as pyo
from pyomo.environ import (
    ConcreteModel,
    Var,
    Param,
    Constraint,
    TransformationFactory,
    SolverFactory,
    Objective,
    minimize,
    value as pyovalue,
    Suffix,
    Expression,
    sin,
    NonNegativeReals,
)



data_ex = {"x0": 1, "x_bounds": [0, 1], "t_range": [0, 1],
           "control_points": {"0": 0, "0.125": 1, "0.25": 1, "0.375": 1, "0.5": 1, "0.625": 1, "0.75": 1,
                              "0.875": 1, "1": 1}, "u_bounds": [0, 1], "a1": 0.5, "a2":0.02, "a3": 0.02, "a4": 0.18, "k1": 0.5, "k2": 0.75}
# Put control input control time points into correct format for two parameter experiment
data_ex["control_points"] = {
    float(k): v for k, v in data_ex["control_points"].items()
}

# Create a two parameterExperiment object; data and discretization information are part
# of the constructor of this object
experiment = SixParameterExperiment(data=data_ex, nfe=10, ncp=3)

# Use a central difference, with step size 1e-3
fd_formula = "central"
step_size = 1e-3


objective_option = "determinant"
scale_nominal_param_value = True

doe_obj = DesignOfExperiments(
    experiment,
    fd_formula=fd_formula,
    step=step_size,
    objective_option=objective_option,
    scale_constant_value=1,
    scale_nominal_param_value=scale_nominal_param_value,
    prior_FIM=None,
    jac_initial=None,
    fim_initial=None,
    L_diagonal_lower_bound=1e-7,
    solver=SolverFactory('IPOPT', options={'linear_solver': 'mumps'}),#, 'print_options_documentation' : 'yes', 'print_level': 5}),
    tee= True,
    get_labeled_model_args=None,
    _Cholesky_option=True,
    _only_compute_fim_lower=True,
)



doe_obj.run_doe()

# Print out a results summary
print("Optimal experiment values: ")
print(
    "\tInitial state: {:.2f}".format(
        doe_obj.results["Experiment Design"][0]
    )
)
print(
    ("\t Control input values: [" + "{:.2f}, " * 8 + "{:.2f}]").format(
        *doe_obj.results["Experiment Design"][1:]
    )
)
print("FIM at optimal design:\n {}".format(np.array(doe_obj.results["FIM"])))
print(
    "Objective value at optimal design: {:.2f}".format(
        pyo.value(doe_obj.model.objective)
    )
)

print(doe_obj.results["Experiment Design Names"])

#print(sorted(doe_obj.results.keys()))

print("Solve time (s):", doe_obj.results["Solve Time"])
print("Build time (s):", doe_obj.results["Build Time"])
print("Initialization time (s):", doe_obj.results["Initialization Time"])
print("Total wall time (s):", doe_obj.results["Wall-clock Time"])

###################
# End optimal DoE



Ipopt 3.13.2: linear_solver=mumps


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        comp

/opt/anaconda3/envs/pyomo/lib/python3.10/site-packages/pyomo/contrib/doe/doe.py:487: RuntimeWarning: invalid value encountered in log10
  self.results["log10 E-opt"] = np.log10(min(np.linalg.eig(fim_local)[0]))


IndexError: Replacement index 8 out of range for positional args tuple

In [ ]:
m.pprint()